# 11 — RAG Evaluation

**FinRAG learning series · Notebook 11 of 14**

## Learning objectives
1. Define retrieval and answer metrics: hit rate, MRR, context precision, context recall, answer correctness, citation
   correctness, faithfulness, latency.
2. Read stored evaluation runs and distinguish **previous** from **new** results.
3. Run a new retrieval evaluation with the production retriever.
4. Understand the difference between a **regression test** and an **independent benchmark**.

## Prerequisites
Notebook 10. The gold file `docs/evaluation/gold_rag_public_annual_report.jsonl` and the matching public document uploaded and
processed (see `docs/evaluation/README.md`), or your own gold file for your own documents.

## Metrics
| Metric | Question it answers | How FinRAG computes it |
|---|---|---|
| Retrieval hit rate / context recall | Did we retrieve the passage that contains the answer? | expected page/sheet/row ∈ retrieved passages |
| MRR | How high was the first relevant passage? | mean of 1/rank of the first relevant passage |
| Context precision | How much of the context is relevant? | relevant retrieved passages / retrieved passages |
| Answer correctness | Is the answer right? | all expected numbers present in the answer |
| Citation correctness | Does a cited source support it? | a cited source contains the expected numbers |
| Faithfulness | Is every number in the answer backed by a source? | no unsupported numbers (verification record) |
| Latency | How long did it take? | wall-clock per question |

"Relevant" here means *on the expected page* — a simple, transparent proxy. It undercounts relevance when the same fact is printed on
several pages.

In [1]:
# Setup: make the production backend importable and keep secrets out of outputs
import sys, os
sys.path.insert(0, os.path.abspath("."))      # run notebooks from the notebooks/ folder
import finrag_nb as nb                          # helper: paths, safe settings, DB session
print("Backend package:", nb.BACKEND_DIR)
print("Documents folder:", nb.DOCS_DIR)

Backend package: /home/user/Ready_To_Deploy/backend
Documents folder: /opt/finrag_testdocs


## 1. Previous results stored in the database

In [2]:
import pandas as pd
from app.models import EvaluationRun
db = nb.db_session()
runs = db.query(EvaluationRun).order_by(EvaluationRun.started_at).all()
rows = []
for r in runs:
    m = r.metrics or {}
    if r.kind == "rag":
        key = {k: m.get(k) for k in ("questions", "answer_accuracy", "citation_correctness", "retrieval_hit_rate", "faithfulness", "errors", "latency_ms_p50")}
    elif r.kind == "finetune":
        key = {"base_fully_correct": m.get("base", {}).get("fully_correct"), "lora_fully_correct": m.get("lora", {}).get("fully_correct")}
    elif r.kind == "extraction":
        key = {"accounting_identity_pass_rate": m.get("accounting_identity_pass_rate"), "identity_check_pass_rate": m.get("identity_check_pass_rate")}
    else:
        key = {k: m.get(k) for k in ("items", "accuracy")}
    rows.append({"started (UTC)": r.started_at.strftime("%Y-%m-%d %H:%M"), "kind": r.kind, "status": r.status, **key})
print("PREVIOUS RESULTS (recorded before this notebook was run):")
pd.DataFrame(rows)

PREVIOUS RESULTS (recorded before this notebook was run):


,started (UTC),kind,status,accounting_identity_pass_rate,identity_check_pass_rate,items,accuracy,base_fully_correct,lora_fully_correct,questions,answer_accuracy,citation_correctness,retrieval_hit_rate,faithfulness,errors,latency_ms_p50
0,2026-11-23 19:37,extraction,completed,NaN,0.8033,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-11-23 19:37,extraction_gold,failed,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-11-23 19:38,extraction,completed,0.9651,0.8033,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-11-23 19:38,extraction_gold,completed,NaN,NaN,15.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-11-23 20:13,finetune,completed,NaN,NaN,NaN,NaN,0.0,0.6929,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2026-11-23 23:58,rag,completed,NaN,NaN,NaN,NaN,NaN,NaN,8.0,0.750,0.750,1.0,1.000,0.0,67732.0
6,2026-11-24 00:09,rag,completed,NaN,NaN,NaN,NaN,NaN,NaN,8.0,0.875,0.875,1.0,1.000,0.0,59004.0
7,2026-11-24 00:26,rag,completed,NaN,NaN,NaN,NaN,NaN,NaN,8.0,0.125,0.125,1.0,0.125,7.0,1356.0
8,2026-11-24 00:31,rag,completed,NaN,NaN,NaN,NaN,NaN,NaN,8.0,1.000,1.000,1.0,1.000,0.0,54703.0


The four `rag` runs used the **same 8 questions**; runs 2–4 came after fixes motivated by earlier failures, and run 3 failed because the
LLM server was killed for lack of memory (7 errors, see `docs/EVALUATION.md`). Run 4's 8/8 is therefore a **regression result**, not
evidence of 100% accuracy in general.

## 2. Answer-level details of the latest completed RAG run

In [3]:
latest = [r for r in runs if r.kind == "rag" and r.status == "completed" and (r.metrics or {}).get("errors") == 0][-1]
print("run", str(latest.id)[:8], "from", latest.started_at.strftime("%Y-%m-%d %H:%M UTC"), "| model:", latest.model_info.get("llm"))
pd.DataFrame([{"question": d["question"][:60], "correct": d["answer_correct"], "citation ok": d["citation_correct"],
               "faithful": d["faithful"], "latency s": round(d["latency_ms"] / 1000, 1), "answer": d["answer"][:90]} for d in latest.details])

run 11592a67 from 2026-11-24 00:31 UTC | model: qwen3-4b-q4_k_m


,question,correct,citation ok,faithful,latency s,answer
0,What was the consolidated revenue from operati...,True,True,True,52.9,The consolidated revenue from operations for 2...
1,What were the consolidated total assets as at ...,True,True,True,45.5,The consolidated total assets as at 31st March...
2,What was the consolidated profit for the year ...,True,True,True,52.8,The consolidated profit for the year 2024-25 w...
3,What were consolidated cash and cash equivalen...,True,True,True,56.5,The consolidated cash and cash equivalents as ...
4,What was the consolidated basic earnings per s...,True,True,True,65.5,The consolidated basic earnings per share for ...
5,What were the consolidated current borrowings ...,True,True,True,57.7,The consolidated current borrowings as at 31st...
6,What were the consolidated finance costs for 2...,True,True,True,68.9,The consolidated finance costs for 2024-25 wer...
7,What was the standalone revenue from operation...,True,True,True,47.5,The standalone revenue from operations for 202...


## 3. New measurement: retrieval quality with the production retriever
This runs only retrieval (no LLM), so it is fast and deterministic. It is a **new result produced in this notebook**, on the same gold
questions — i.e. still a regression check.

In [4]:
import json, time
from app.models import Document
from app.rag import retriever
user = nb.notebook_user(db)
gold = [json.loads(l) for l in open(nb.PROJECT_DIR / "docs/evaluation/gold_rag_public_annual_report.jsonl")]
rows = []
for g in gold:
    docs = db.query(Document).filter(Document.owner_id == user.id, Document.original_filename.in_(g["document_filenames"])).all()
    t = time.time()
    passages = retriever.retrieve(db, user.id, g["question"], docs)
    secs = time.time() - t
    page = g["expected_location"]["page"]
    rel = [(p.chunk.location.get("page") or 0) <= page <= (p.chunk.location.get("page_end") or p.chunk.location.get("page") or 0) for p in passages]
    first = next((i + 1 for i, x in enumerate(rel) if x), None)
    rows.append({"question": g["question"][:55], "expected page": page, "hit": any(rel), "rank of first relevant": first,
                 "context precision": round(sum(rel) / len(rel), 2) if rel else 0.0, "retrieval s": round(secs, 1)})
res = pd.DataFrame(rows)
res

,question,expected page,hit,rank of first relevant,context precision,retrieval s
0,What was the consolidated revenue from operati...,100,True,2.0,0.2,6.4
1,What were the consolidated total assets as at ...,99,True,5.0,0.2,1.5
2,What was the consolidated profit for the year ...,100,True,3.0,0.2,1.3
3,What were consolidated cash and cash equivalen...,99,True,3.0,0.2,0.9
4,What was the consolidated basic earnings per s...,100,True,2.0,0.4,0.7
5,What were the consolidated current borrowings ...,99,False,NaN,0.0,1.0
6,What were the consolidated finance costs for 2...,100,True,2.0,0.2,1.3
7,What was the standalone revenue from operation...,58,True,3.0,0.2,1.0


In [5]:
mrr = sum(1 / r for r in res["rank of first relevant"] if pd.notna(r)) / len(res)   # a miss contributes 0
print(f"NEW RESULT ({time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())}), {len(res)} gold questions, top-{len(passages)} passages:")
print(f"  context recall / hit rate : {res['hit'].mean():.1%}")
print(f"  MRR                       : {mrr:.3f}")
print(f"  mean context precision    : {res['context precision'].mean():.1%}")
print(f"  mean retrieval latency    : {res['retrieval s'].mean():.1f}s (CPU)")
db.close()

NEW RESULT (2026-11-24 04:13 UTC), 8 gold questions, top-5 passages:
  context recall / hit rate : 87.5%
  MRR                       : 0.338
  mean context precision    : 20.0%
  mean retrieval latency    : 1.8s (CPU)


## 4. Regression tests vs independent evaluation
* A **regression test** re-runs a fixed set after changes to make sure nothing broke. If you used its failures to design the fix
  (as we did in `docs/EVALUATION.md`), its score is optimistic.
* An **independent evaluation** uses questions written *before* looking at the system's behaviour, ideally on **documents not used
  during development**, and is run once per system version.

### How to build an independent gold set for your documents
1. Pick documents you have not used while developing or tuning.
2. Write 30–100 questions with expected numbers and the page/sheet/row where you found them, before running FinRAG.
3. Run the `rag` evaluation once from the Evaluation page; report the numbers with the date, model and documents.
4. Do not change prompts or retrieval in response to that set; create a new set for the next round.

## Interpretation
* The stored runs show the history honestly, including the failed run.
* The new retrieval-only measurement tells you whether the right page reaches the LLM at all — when retrieval misses, no prompt can fix
  the answer. Context precision shows how much of the prompt budget is spent on relevant pages.

## Common errors and troubleshooting
| Symptom | Cause | Fix |
|---|---|---|
| Gold questions find no documents | Filenames in the gold file differ from uploads | Upload with the original filename, or use `document_ids` |
| All hits false | Expected page refers to printed page instead of PDF page | Gold files use **PDF** page numbers |
| Evaluation run stuck "running" | LLM very slow or down | Check the Status page; runs record errors instead of fabricating |

## Practical exercises
1. Write 10 independent questions for one of your documents and run a `rag` evaluation from the app.
2. Change `retrieval_top_k` (environment variable `RETRIEVAL_TOP_K`) and re-run section 3. Does recall change?
3. Add a `sheet` expected location for a spreadsheet question and evaluate it.

## Summary
You read previous evaluation results with their context, produced a new retrieval measurement, and learnt how to turn a regression set
into an independent benchmark.

## Further learning
* RAGAS metrics (context precision/recall, faithfulness): https://docs.ragas.io/
* FinRAG source: `backend/app/evaluation/runner.py`; results: `docs/EVALUATION.md`
* Next: **12 — Financial analytics**.